## using COlab GPU tp tune the model test if you are connected to GPU or not

In [3]:
import os

gpu_env = os.environ.get("COLAB_GPU", "")

if gpu_env.isdigit() and int(gpu_env) > 0:
    print("✅ Connected to Google Colab GPU")
elif "COLAB_RELEASE_TAG" in os.environ:
    print("☁️ Connected to Google Colab (CPU mode)")
else:
    print("💻 Running on Local Machine")


✅ Connected to Google Colab GPU


# Phase 3: Fine-Tuning YOLO & Moderate Class Optimization

## Objectives
1. Fine-tune YOLOv8 to improve vehicle detection recall.
2. Re-calibrate traffic-state thresholds to improve Moderate class F1-score.

This phase uses a newly created train/val/test split.

In [7]:
from pathlib import Path
import random, shutil
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Step 1: Create New Train/Val/Test Split (Phase 3)

Current dataset structure:
- train/images
- train/labels
- valid/images
- valid/labels

We combine train + valid,
then create a new 70/15/15 split.

In [10]:
from pathlib import Path
import random, shutil

# YOUR ACTUAL PATH
DATASET_ROOT = Path('../../data/Vehicle_Detection_Image_Dataset')

TRAIN_IMAGES = DATASET_ROOT / 'train' / 'images'
TRAIN_LABELS = DATASET_ROOT / 'train' / 'labels'
VALID_IMAGES = DATASET_ROOT / 'valid' / 'images'
VALID_LABELS = DATASET_ROOT / 'valid' / 'labels'

# New Phase 3 dataset folder
OUT_ROOT = Path('../../data/Vehicle_Detection_Image_Dataset_Phase3')

SEED = 42
TRAIN_RATIO = 0.60
VAL_RATIO = 0.20

# Collect all images from train + valid
all_images = list(TRAIN_IMAGES.glob('*')) + list(VALID_IMAGES.glob('*'))

random.seed(SEED)
random.shuffle(all_images)

n = len(all_images)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_imgs = all_images[:n_train]
val_imgs = all_images[n_train:n_train+n_val]
test_imgs = all_images[n_train+n_val:]

print(f"Total images: {n}")
print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")


def copy_pair(img_path, split):
    # Determine label location
    if "train" in str(img_path):
        label_path = TRAIN_LABELS / (img_path.stem + ".txt")
    else:
        label_path = VALID_LABELS / (img_path.stem + ".txt")

    dst_img_dir = OUT_ROOT / split / "images"
    dst_lab_dir = OUT_ROOT / split / "labels"

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lab_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(img_path, dst_img_dir / img_path.name)
    shutil.copy2(label_path, dst_lab_dir / label_path.name)


for split_name, imgs in [("train", train_imgs),
                         ("val", val_imgs),
                         ("test", test_imgs)]:
    for img in imgs:
        copy_pair(img, split_name)

print("Fine tune split created successfully.")

Total images: 0
Train: 0, Val: 0, Test: 0
Fine tune split created successfully.


In [12]:
from pathlib import Path

DATASET_ROOT = Path('../../data/Vehicle_Detection_Image_Dataset')
TRAIN_IMAGES = DATASET_ROOT / 'train' / 'images'
VALID_IMAGES = DATASET_ROOT / 'valid' / 'images'

print("Notebook CWD:", Path().resolve())
print("DATASET_ROOT:", DATASET_ROOT.resolve(), "exists=", DATASET_ROOT.exists())
print("TRAIN_IMAGES:", TRAIN_IMAGES.resolve(), "exists=", TRAIN_IMAGES.exists())
print("VALID_IMAGES:", VALID_IMAGES.resolve(), "exists=", VALID_IMAGES.exists())

Notebook CWD: /content
DATASET_ROOT: /data/Vehicle_Detection_Image_Dataset exists= False
TRAIN_IMAGES: /data/Vehicle_Detection_Image_Dataset/train/images exists= False
VALID_IMAGES: /data/Vehicle_Detection_Image_Dataset/valid/images exists= False


In [43]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [49]:
!ls "/content/drive/.Encrypted/MyDrive/Drexel_Program/CS 614-686_Applications of ML/Final Project/TrafficVision/data/Vehicle_Detection_Image_Dataset/train/images"

DATASET_ROOT_STR = "/content/drive/.Encrypted/MyDrive/Drexel_Program/CS 614-686_Applications of ML/Final Project/TrafficVision/data/Vehicle_Detection_Image_Dataset"
!find "{DATASET_ROOT_STR}" -type f | head -n 20

TRAIN_IMAGES_STR = DATASET_ROOT_STR + "/train/images"
VALID_IMAGES_STR = DATASET_ROOT_STR + "/valid/images"

!ls -lah "{TRAIN_IMAGES_STR}" | head -n 50
!ls -lah "{VALID_IMAGES_STR}" | head -n 50

total 0
total 0


In [29]:
from pathlib import Path
import random, shutil

# Your exact dataset path in Colab Drive
DATASET_ROOT = Path("/content/drive/.Encrypted/MyDrive/Drexel_Program/CS 614-686_Applications of ML/Final Project/TrafficVision/data/Vehicle_Detection_Image_Dataset")

TRAIN_IMAGES = DATASET_ROOT / "train" / "images"
TRAIN_LABELS = DATASET_ROOT / "train" / "labels"
VALID_IMAGES = DATASET_ROOT / "valid" / "images"
VALID_LABELS = DATASET_ROOT / "valid" / "labels"

# Output fine tune dataset (new split) - keep it near original dataset
OUT_ROOT = Path("/content/drive/.Encrypted/MyDrive/Drexel_Program/CS 614-686_Applications of ML/Final Project/TrafficVision/data/Vehicle_Detection_Image_Dataset_Phase3")

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# --- sanity checks
print("TRAIN_IMAGES exists:", TRAIN_IMAGES.exists())
print("VALID_IMAGES exists:", VALID_IMAGES.exists())

def collect_images(folder: Path):
    exts = {".jpg",".jpeg",".png",".bmp",".webp"}
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts]

train_imgs = collect_images(TRAIN_IMAGES)
valid_imgs = collect_images(VALID_IMAGES)
all_images = train_imgs + valid_imgs

print("Train images:", len(train_imgs))
print("Valid images:", len(valid_imgs))
print("Total images:", len(all_images))

if len(all_images) < 3:
    raise ValueError("Need at least 3 images total to create train/val/test split.")

# --- shuffle
random.seed(SEED)
random.shuffle(all_images)

n = len(all_images)

# --- guarantee at least 1 in val/test
n_val  = max(1, int(round(n * VAL_RATIO)))
n_test = max(1, int(round(n * TEST_RATIO)))
n_train = n - n_val - n_test

# if rounding makes train 0, steal back (rare)
while n_train < 1:
    if n_val > 1:
        n_val -= 1
    elif n_test > 1:
        n_test -= 1
    else:
        raise ValueError("Not enough images to keep train/val/test >= 1 each.")
    n_train = n - n_val - n_test

train_split = all_images[:n_train]
val_split   = all_images[n_train:n_train+n_val]
test_split  = all_images[n_train+n_val:n_train+n_val+n_test]

print("Split sizes -> Train:", len(train_split), "Val:", len(val_split), "Test:", len(test_split))

def copy_pair(img_path: Path, split: str):
    # Determine where the label lives (train or valid original folder)
    if "train/images" in str(img_path):
        label_path = TRAIN_LABELS / (img_path.stem + ".txt")
    else:
        label_path = VALID_LABELS / (img_path.stem + ".txt")

    if not label_path.exists():
        raise FileNotFoundError(f"Missing label for {img_path.name}: expected {label_path}")

    dst_img_dir = OUT_ROOT / split / "images"
    dst_lab_dir = OUT_ROOT / split / "labels"
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lab_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(img_path, dst_img_dir / img_path.name)
    shutil.copy2(label_path, dst_lab_dir / label_path.name)

# Always create the folders
for split in ["train", "val", "test"]:
    (OUT_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

# Copy files
for split_name, split_data in [("train", train_split), ("val", val_split), ("test", test_split)]:
    for img in split_data:
        copy_pair(img, split_name)

print("Fine tune dataset created at:")
print(OUT_ROOT)

TRAIN_IMAGES exists: True
VALID_IMAGES exists: True
Train images: 0
Valid images: 0
Total images: 0


ValueError: Need at least 3 images total to create train/val/test split.